# Window Parameter Grid: Precision/Recall/F1 vs. cost, pen, min_size, width (red/snr3)

Single-method, single-condition diagnostic: fix `ruptures` algorithm to **Window**, run it on
the **red/snr3** noise condition only (`noise_sensitivity_data/noise_red_snr3.npz`, 20 seeds), and
grid-search `cost` x `pen` x `min_size` x `width` for all 9 `ruptures` cost functions (`l1`, `l2`,
`rbf`, `normal`, `ar`, `linear`, `rank`, `clinear`, `mahalanobis`). Mirrors
`binseg_grid_red_snr3.ipynb` / `pelt_grid_red_snr3.ipynb` / `bottomup_grid_red_snr3.ipynb` -- same
data, same ground truth, same methodology, with one addition below.

**What's new here.** `Window` requires a `width` parameter that Binseg/Pelt/BottomUp don't have --
the half-window length used to build the discrepancy curve that candidate breakpoints get scored
from. It is not the same thing as `min_size` (a post-hoc floor on segment length, applied after
breakpoints are already candidates): `width` sets the resolution the detector sees the signal at
*before* any breakpoint is proposed. So the grid search below is a genuine 3-way search over
`pen x min_size x width`, jointly, per cost function -- not `pen x min_size` with width bolted on
afterward.

**Each cost function has its own cells (Section 7), same as before** -- one to define its grid and
run it, one to plot it. Run/tweak/rerun any cost independently without touching the others; nothing
downstream (Section 8/9) depends on run order.

**Plotting, since we now have three params instead of two.** Rather than force `pen`, `min_size`,
and `width` into a single illegible plot, each cost function gets five diagnostics:

1. **Train surface** -- `pen x width` response surface, sliced at the *winning* `min_size` (the
   same 3D-surface convention as the other notebooks, just with `width` swapped in for the axis
   that is now fixed).
2. **Test PR curve** -- unchanged from Binseg/Pelt/BottomUp: pen swept at the winning
   `min_size`/`width`, test seeds only.
3. **Width profile** -- best-over-pen train f1 at each tested `width` (min_size fixed at the
   winner). Same job as the `on_edge` check, but visual: confirms the winning `width` sits inside
   the tested range rather than at its boundary.
4. **Parallel coordinates** -- every `(pen, min_size, width)` combo in the train grid as one line,
   colored by train f1, winner highlighted. Shows all three params (not just the two the surface
   plot slices to) in one view.
5. **3D scatter** -- the full `(pen, min_size, width)` train grid as points colored by f1, winner
   starred. A sanity check that slicing the surface at the winning `min_size` isn't hiding a
   better region at some other `min_size` -- if the surface (1) already converges cleanly, this
   mostly just confirms it.

**Seed budget.** `rbf` is ~100-1000x slower per call than every other cost function regardless of
algorithm, so it gets a smaller seed budget (10 seeds: 7 train / 3 test) to stay tractable.
Everything else uses the full budget (20 seeds: 14 train / 6 test). Edit `HEAVY_COSTS` below to
reclassify. `rbf` also gets a coarser `width` grid (4 points instead of 6) -- the width axis
multiplies grid size on top of the existing pen x min_size grid, and `rbf` is already the slowest
cell by a wide margin.

**`width` ranges below are a first pass, not yet tuned per cost function** -- unlike `pen`/
`min_size`, which mirror Binseg/Pelt/BottomUp's already-converged ranges, there is no prior run to
inherit a `width` range from. Widen a cost's `WIDTHS_<cost>` and rerun its two cells if `on_edge`
fires on `width` specifically.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ammonyte as amt
from ammonyte.utils.metrics import evaluate_detection
from joblib import Parallel, delayed
from mpl_toolkits.mplot3d import Axes3D

## Section 1 -- Configuration

In [ ]:
DATA_PATH   = "../../noise_sensitivity_data/noise_red_snr3.npz"
TOLERANCE   = 0.5   # +/- 0.5 kyr matching tolerance, matches cv_noise_sensitivity.py
RANDOM_SEED = 42
ALGO = 'Window'
N_JOBS = -1   # CARC run: use all allocated cores; lower this if running locally

# rbf is ~100-1000x slower per call than every other cost function regardless of
# algorithm -- it gets a smaller seed budget so the grid stays tractable.
# Edit this set to reclassify a cost function as heavy.
HEAVY_COSTS = {'rbf'}
LIGHT_BUDGET = dict(pool_size=20, n_train=14, n_test=6)
HEAVY_BUDGET = dict(pool_size=10, n_train=7, n_test=3)

MIN_SIZES = [2, 3, 5, 7, 10, 15, 20, 30, 40, 50]   # shared default -- override per cost if needed
WIDTHS    = [5, 10, 20, 40, 80, 150]         # shared default, untuned -- override per cost if needed


def logrange(lo, hi, n):
    return list(np.unique(np.round(np.logspace(np.log10(lo), np.log10(hi), n), 3)))

## Section 2 -- Data and ground truth

In [ ]:
def transition_times_from_states(time, states):
    """True transition times (both directions) from a state-label array.

    Mirrors the same helper in cv_noise_sensitivity.py / Noise_Sensitivity_Synthetic_Data.ipynb:
    any stadial/interstadial change counts as ground truth, regardless of direction (Melcher25's
    transitions are symmetric).
    """
    change_idx = np.where(np.diff(states) != 0)[0] + 1
    return time[change_idx]


d = np.load(DATA_PATH)
time_arr   = d['time']
db_noisy   = d['db_noisy']
state_true = d['state_true']
n_seeds    = db_noisy.shape[0]

gt_times = [transition_times_from_states(time_arr, state_true[s]) for s in range(n_seeds)]
n_transitions = [len(g) for g in gt_times]
print(f"Loaded {DATA_PATH}: {n_seeds} seeds, {len(time_arr)} points/series")
print(f"Ground-truth transitions per seed: min={min(n_transitions)}, max={max(n_transitions)}, "
      f"mean={np.mean(n_transitions):.1f}")

## Section 3 -- Detection wrapper and per-combo scoring

In [ ]:
def _clean_times(times):
    times = np.asarray(times, dtype=float)
    return times[~np.isnan(times)]


def detect_window(time, value, cost, pen, min_size, width, jump=1):
    series = amt.Series(time=time, value=value, label='grid', verbose=False)
    transitions = series.ruptures(algo=ALGO, cost=cost, pen=pen, min_size=min_size, jump=jump, width=width)
    return _clean_times(transitions.jump_times)


def _score_one(params, seed):
    try:
        detected = detect_window(time_arr, db_noisy[seed], **params)
    except Exception:
        return dict(precision=0.0, recall=0.0, f1=0.0, false_positives=np.nan)
    m = evaluate_detection(detected=detected, ground_truth=gt_times[seed], tolerance=TOLERANCE)
    return dict(precision=m.precision, recall=m.recall, f1=m.f1_score, false_positives=m.false_positives)

## Section 4 -- Train/test split utilities

In [ ]:
def seed_pool(n_seeds_total, pool_size, random_state=RANDOM_SEED):
    """Fixed subsample of `pool_size` seed indices out of n_seeds_total."""
    rng = np.random.RandomState(random_state)
    return np.sort(rng.choice(n_seeds_total, size=pool_size, replace=False))


def outer_split(pool, n_train, n_test, random_state=RANDOM_SEED):
    """Split `pool` into disjoint train/test seed-index arrays, once."""
    rng = np.random.RandomState(random_state)
    shuffled = rng.permutation(pool)
    train_idx = np.sort(shuffled[:n_train])
    test_idx = np.sort(shuffled[n_train:n_train + n_test])
    return train_idx, test_idx


def split_for_cost(cost, n_seeds_total, random_state=RANDOM_SEED):
    budget = HEAVY_BUDGET if cost in HEAVY_COSTS else LIGHT_BUDGET
    pool = seed_pool(n_seeds_total, budget['pool_size'], random_state=random_state)
    return outer_split(pool, budget['n_train'], budget['n_test'], random_state=random_state)

## Section 5 -- Grid search (train pool) + held-out evaluation (test pool)

In [ ]:
def grid_search_train(param_grid, train_idx):
    """Score every (param, seed in train_idx) combo, aggregate per param, pick the
    winner by mean f1 over the train pool only."""
    jobs = [(pi, params, s) for pi, params in enumerate(param_grid) for s in train_idx]
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(params, s) for pi, params, s in jobs
    )
    rows = []
    for (pi, params, s), res in zip(jobs, scored):
        row = dict(params)
        row.update(pi=pi, seed=int(s), split='train', **res)
        rows.append(row)
    train_raw = pd.DataFrame(rows)

    param_cols = list(param_grid[0].keys())
    train_summary = (train_raw.groupby(param_cols)
                      .agg(mean_precision=('precision', 'mean'), std_precision=('precision', 'std'),
                           mean_recall=('recall', 'mean'), std_recall=('recall', 'std'),
                           mean_f1=('f1', 'mean'), std_f1=('f1', 'std'),
                           mean_fp=('false_positives', 'mean'))
                      .reset_index())

    best_row = train_summary.loc[train_summary['mean_f1'].idxmax()]
    best_params = {c: best_row[c] for c in param_cols}
    return best_params, best_row, train_summary, train_raw


def evaluate_test(best_params, test_idx):
    """Apply best_params to the untouched-until-now test pool, once."""
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(best_params, s) for s in test_idx
    )
    rows = []
    for s, res in zip(test_idx, scored):
        row = dict(best_params)
        row.update(seed=int(s), split='test', **res)
        rows.append(row)
    test_raw = pd.DataFrame(rows)
    return (test_raw['precision'].mean(), test_raw['recall'].mean(),
            test_raw['f1'].mean(), test_raw['false_positives'].mean(), test_raw)


def pen_sweep_test(cost, pen_grid, min_size, test_idx, extra_params=None):
    """Sweep pen at the winning min_size (and any other winning params, e.g. width/jump,
    passed via extra_params), test seeds only -- precision/recall tradeoff curve
    for this cost function (the pen-analog of a PR curve)."""
    extra_params = extra_params or {}
    jobs = [(pen, s) for pen in pen_grid for s in test_idx]
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(dict(cost=cost, pen=pen, min_size=min_size, **extra_params), s)
        for pen, s in jobs
    )
    rows = []
    for (pen, s), res in zip(jobs, scored):
        rows.append(dict(cost=cost, pen=pen, min_size=min_size, seed=int(s), split='test_sweep', **res))
    sweep_raw = pd.DataFrame(rows)
    sweep_summary = (sweep_raw.groupby('pen')
                      .agg(mean_precision=('precision', 'mean'), mean_recall=('recall', 'mean'),
                           mean_f1=('f1', 'mean'), mean_fp=('false_positives', 'mean'))
                      .reset_index())
    return sweep_summary, sweep_raw


results = {}   # every cost's run_one_cost result lands here, keyed by cost name


def run_one_cost(cost, pens, min_sizes, widths, jumps=None):
    t0 = time.perf_counter()
    train_idx, test_idx = split_for_cost(cost, n_seeds)
    budget = 'HEAVY' if cost in HEAVY_COSTS else 'light'

    if jumps is None:
        param_grid = [{'cost': cost, 'pen': pen, 'min_size': ms, 'width': w}
                      for pen in pens for ms in min_sizes for w in widths]
    else:
        param_grid = [{'cost': cost, 'pen': pen, 'min_size': ms, 'width': w, 'jump': j}
                      for pen in pens for ms in min_sizes for w in widths for j in jumps]

    best_params, train_row, train_summary, train_raw = grid_search_train(param_grid, train_idx)
    test_p, test_r, test_f1, test_fp, test_raw = evaluate_test(best_params, test_idx)

    sweep_extra = {'width': best_params['width']}
    if jumps is not None:
        sweep_extra['jump'] = best_params['jump']
    sweep_summary, sweep_raw = pen_sweep_test(cost, pens, best_params['min_size'], test_idx,
                                               extra_params=sweep_extra)

    on_edge = (best_params['pen'] in (min(pens), max(pens))
               or best_params['min_size'] in (min(min_sizes), max(min_sizes))
               or best_params['width'] in (min(widths), max(widths)))

    conclusion_row = dict(
        cost=cost, best_pen=best_params['pen'], best_min_size=best_params['min_size'],
        best_width=best_params['width'], best_jump=best_params.get('jump', 1),
        train_f1=train_row['mean_f1'], train_precision=train_row['mean_precision'],
        train_recall=train_row['mean_recall'],
        test_f1=test_f1, test_precision=test_p, test_recall=test_r, test_fp=test_fp,
        pen_range=f"{min(pens):.3g}-{max(pens):.3g}",
        min_size_range=f"{min(min_sizes):.0f}-{max(min_sizes):.0f}",
        width_range=f"{min(widths):.0f}-{max(widths):.0f}",
        on_edge=on_edge, seed_budget=budget,
        n_train_seeds=len(train_idx), n_test_seeds=len(test_idx),
        train_seeds=",".join(map(str, train_idx)), test_seeds=",".join(map(str, test_idx)),
    )

    result = dict(train_idx=train_idx, test_idx=test_idx, best_params=best_params,
                  train_summary=train_summary, train_raw=train_raw, test_raw=test_raw,
                  sweep_summary=sweep_summary, sweep_raw=sweep_raw, conclusion_row=conclusion_row)

    elapsed = time.perf_counter() - t0
    edge_note = "  <-- ON GRID EDGE" if on_edge else ""
    print(f"[{budget:5s}] {cost:12s}: pen={best_params['pen']:>8.3g} min_size={best_params['min_size']:>3.0f} "
          f"width={best_params['width']:>4.0f} jump={best_params.get('jump', 1)}  "
          f"train_f1={train_row['mean_f1']:.3f}  test_f1={test_f1:.3f}  ({elapsed:.0f}s){edge_note}")

    results[cost] = result
    return result

## Section 6 -- Plot helpers

In [ ]:
def plot_train_surface(cost, result):
    """3D response surface of mean_f1 over the (pen, width) TRAIN grid, sliced at the
    winning min_size -- this is the surface the winning combo was actually picked from."""
    summary = result['train_summary']
    best_pen, best_width = result['best_params']['pen'], result['best_params']['width']
    best_ms = result['best_params']['min_size']

    summary = summary[summary['min_size'] == best_ms]
    if 'jump' in summary.columns:
        summary = summary[summary['jump'] == result['best_params'].get('jump', 1)]

    pivot = summary.pivot(index='width', columns='pen', values='mean_f1')
    pen_vals, width_vals = pivot.columns.values, pivot.index.values
    X, Y = np.meshgrid(pen_vals, width_vals)
    Z = pivot.values

    fig = plt.figure(figsize=(6, 5.5))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(X, Y, Z, cmap='viridis', vmin=0, vmax=np.nanmax(Z),
                     edgecolor='k', linewidth=0.2, antialiased=True)

    zi, xi = list(width_vals).index(best_width), list(pen_vals).index(best_pen)
    ax.scatter([best_pen], [best_width], [Z[zi, xi]], color='red', s=70, marker='*', zorder=10)

    on_edge = result['conclusion_row']['on_edge']
    flag = 'edge -- NOT converged' if on_edge else 'interior -- converged'
    ax.set_title(f"{cost}: train mean_f1 (min_size={best_ms:.0f} fixed)\nbest={Z[zi, xi]:.3f} @ pen={best_pen:.3g}, "
                 f"width={best_width:.0f}\n({flag})",
                 fontsize=9, color='#b3261e' if on_edge else '#1a1a1a')
    ax.set_xlabel('pen', fontsize=8)
    ax.set_ylabel('width', fontsize=8)
    ax.set_zlabel('mean_f1 (train)', fontsize=8)
    ax.set_zlim(0, max(np.nanmax(Z) * 1.05, 0.05))
    ax.view_init(elev=25, azim=-60)
    fig.suptitle(f"cost='{cost}': TRAIN response surface (pen x width, min_size fixed at winner)", fontsize=10)
    plt.tight_layout()
    fig.savefig(f"window_grid_red_snr3_{cost}_surface.png", dpi=200, bbox_inches='tight')
    plt.show()


def plot_test_pr(cost, result):
    """Test-set precision/recall tradeoff (pen swept at the winning min_size/width/jump).
    Each point is labeled with its pen value so you can tell exactly which point corresponds
    to which pen -- points are connected in order of increasing pen (pandas groupby sorts by
    the group key by default), not sorted by recall/precision, so the line can double back if
    the pen/precision relationship isn't perfectly monotonic on this seed sample.

    Labels cycle through several offset directions and draw a thin leader line back to their
    point -- when points sit close together the labels fan out instead of stacking on top of
    each other, and the leader line keeps each label unambiguously tied to its point.
    """
    sweep, row = result['sweep_summary'], result['conclusion_row']
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.plot(sweep['mean_recall'], sweep['mean_precision'], color='#2a78d6',
             linewidth=1.5, marker='o', markersize=4, zorder=2)

    offset_cycle = [(10, 10), (10, -16), (-38, 10), (-38, -16), (10, 26), (-38, -32)]
    for i, (_, r) in enumerate(sweep.iterrows()):
        dx, dy = offset_cycle[i % len(offset_cycle)]
        ax.annotate(f"{r['pen']:.3g}", (r['mean_recall'], r['mean_precision']),
                    fontsize=7, color='#555555', xytext=(dx, dy), textcoords='offset points',
                    arrowprops=dict(arrowstyle='-', color='#aaaaaa', linewidth=0.6,
                                     shrinkA=0, shrinkB=3))

    ax.scatter([row['test_recall']], [row['test_precision']], color='#e34948',
               s=90, marker='*', zorder=5, label='selected pen')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('recall')
    ax.set_ylabel('precision')
    ax.set_title(f"{cost}: TEST precision/recall tradeoff (labels = pen value)")
    ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(color='#e1e0d9', linewidth=0.6)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()


def plot_width_profile(cost, result):
    """f1 vs width profile: best-over-pen train f1 at each width (min_size fixed at the
    winner) -- confirms the picked width is not sitting on a grid edge."""
    summary = result['train_summary']
    best_ms = result['best_params']['min_size']
    best_width = result['best_params']['width']
    if 'jump' in summary.columns:
        summary = summary[summary['jump'] == result['best_params'].get('jump', 1)]
    summary = summary[summary['min_size'] == best_ms]

    profile = summary.groupby('width')['mean_f1'].max().reset_index()

    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.plot(profile['width'], profile['mean_f1'], color='#2a78d6', marker='o', markersize=4)
    ax.axvline(best_width, color='#e34948', linestyle='--', linewidth=1,
               label=f'winning width={best_width:.0f}')
    ax.set_xlabel('width')
    ax.set_ylabel('best mean_f1 over pen (train)')
    ax.set_title(f"{cost}: f1 vs width (min_size={best_ms:.0f} fixed)")
    ax.legend(frameon=False, fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(color='#e1e0d9', linewidth=0.6)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()


def plot_parallel_coordinates(cost, result):
    """Parallel-coordinates view of the full (pen, min_size, width) train grid, colored by
    f1 -- complements the pen x width surface by showing all three params (not just the
    winning min_size slice) at once."""
    summary = result['train_summary']
    if 'jump' in summary.columns:
        summary = summary[summary['jump'] == result['best_params'].get('jump', 1)]

    axes_cols = ['pen', 'min_size', 'width']
    df = summary[axes_cols + ['mean_f1']].copy()
    mins, maxs = df[axes_cols].min(), df[axes_cols].max()
    span = (maxs - mins).replace(0, 1)
    scaled = (df[axes_cols] - mins) / span

    fig, ax = plt.subplots(figsize=(6, 4))
    norm = plt.Normalize(df['mean_f1'].min(), df['mean_f1'].max())
    cmap = plt.get_cmap('viridis')
    x = np.arange(len(axes_cols))

    for idx in df['mean_f1'].argsort().values:   # draw best-scoring lines last, on top
        ax.plot(x, scaled.iloc[idx].values, color=cmap(norm(df['mean_f1'].iloc[idx])),
                linewidth=0.8, alpha=0.6)

    best = result['best_params']
    best_scaled = [(best[c] - mins[c]) / span[c] for c in axes_cols]
    ax.plot(x, best_scaled, color='#e34948', linewidth=2.2, marker='*', markersize=12,
            zorder=10, label='winner')

    ax.set_xticks(x)
    ax.set_xticklabels([f"{c}\n[{mins[c]:.3g}-{maxs[c]:.3g}]" for c in axes_cols], fontsize=8)
    ax.set_yticks([])
    ax.set_title(f"{cost}: parallel coordinates (pen, min_size, width), colored by train f1")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, label='mean_f1 (train)', fraction=0.046, pad=0.04)
    ax.legend(frameon=False, fontsize=8, loc='upper right')
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_3d_scatter(cost, result):
    """3D scatter of the full (pen, min_size, width) train grid, colored by f1 -- sanity
    check that slicing the surface at the winning min_size is not hiding a better region."""
    summary = result['train_summary']
    if 'jump' in summary.columns:
        summary = summary[summary['jump'] == result['best_params'].get('jump', 1)]
    best = result['best_params']

    fig = plt.figure(figsize=(6, 5.5))
    ax = fig.add_subplot(111, projection='3d')
    sc = ax.scatter(summary['pen'], summary['min_size'], summary['width'],
                     c=summary['mean_f1'], cmap='viridis', s=25, alpha=0.7)
    ax.scatter([best['pen']], [best['min_size']], [best['width']],
               color='red', s=120, marker='*', zorder=10, label='winner')
    ax.set_xlabel('pen', fontsize=8)
    ax.set_ylabel('min_size', fontsize=8)
    ax.set_zlabel('width', fontsize=8)
    ax.set_title(f"{cost}: full 3D grid, colored by train f1", fontsize=9)
    fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.15, label='mean_f1 (train)')
    ax.legend(frameon=False, fontsize=8)
    ax.view_init(elev=20, azim=-60)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_min_size_profile(cost, result):
    """f1 vs min_size profile: best-over-pen train f1 at each min_size (width fixed at the
    winner) -- shows whether min_size has a real interior optimum or is monotonic/flat,
    i.e. whether the grid search winner is a meaningful signal or noise."""
    summary = result['train_summary']
    best_width = result['best_params']['width']
    best_ms = result['best_params']['min_size']
    if 'jump' in summary.columns:
        summary = summary[summary['jump'] == result['best_params'].get('jump', 1)]
    summary = summary[summary['width'] == best_width]

    profile = summary.groupby('min_size')['mean_f1'].max().reset_index()

    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.plot(profile['min_size'], profile['mean_f1'], color='#2a78d6', marker='o', markersize=4)
    ax.axvline(best_ms, color='#e34948', linestyle='--', linewidth=1,
               label=f'winning min_size={best_ms:.0f}')
    ax.set_xlabel('min_size')
    ax.set_ylabel('best mean_f1 over pen (train)')
    ax.set_title(f"{cost}: f1 vs min_size (width={best_width:.0f} fixed)")
    ax.legend(frameon=False, fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(color='#e1e0d9', linewidth=0.6)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()

    spread = profile['mean_f1'].max() - profile['mean_f1'].min()
    print(f"{cost}: mean_f1 range across tested min_size = {spread:.4f} "
          f"({'looks meaningful' if spread > 0.02 else 'looks like noise -- min_size barely matters'})")

In [ ]:
def plot_pen_profile(cost, result):
    """f1 vs pen profile: best-over-(min_size,width) train f1 at each tested pen -- shows
    whether pen has a real interior optimum, is still climbing at the floor (genuine trend,
    check test_f1 for overfitting), or is flat near the floor (tie -- pushing the range
    lower won't tell you anything new)."""
    summary = result['train_summary']
    best_pen = result['best_params']['pen']
    if 'jump' in summary.columns:
        summary = summary[summary['jump'] == result['best_params'].get('jump', 1)]

    profile = summary.groupby('pen')['mean_f1'].max().reset_index()

    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.plot(profile['pen'], profile['mean_f1'], color='#2a78d6', marker='o', markersize=4)
    ax.axvline(best_pen, color='#e34948', linestyle='--', linewidth=1,
               label=f'winning pen={best_pen:.3g}')
    ax.set_xscale('log')
    ax.set_xlabel('pen (log scale)')
    ax.set_ylabel('best mean_f1 over (min_size, width) (train)')
    ax.set_title(f"{cost}: f1 vs pen")
    ax.legend(frameon=False, fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(color='#e1e0d9', linewidth=0.6)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()

    lowest_two = profile.nsmallest(2, 'pen')['mean_f1'].values
    step = abs(lowest_two[0] - lowest_two[1]) if len(lowest_two) == 2 else float('nan')
    print(f"{cost}: mean_f1 at two lowest tested pens differ by {step:.4f} "
          f"({'still moving -- check test_f1 for overfitting before widening further' if step > 0.005 else 'flat -- floor is a tie, stop widening down'})")

In [ ]:
def pen_1se_selection(cost, result):
    """One-standard-error rule for pen, train-side only (no test touched).

    pen is a regularization strength (smaller pen = less penalty = more breakpoints
    allowed), so picking the single best train mean_f1 is exactly the failure mode that
    makes LASSO/ridge overfit if you don't correct for it: the argmax tends to land on the
    least-regularized (smallest) value because that has the most freedom to fit train-pool
    noise. The 1-SE rule picks the *largest* (most regularized) pen whose mean_f1 is still
    within one standard error of the best -- "as simple as the data can justify" rather than
    "whatever scored highest by chance." This never looks at test_idx.
    """
    summary = result['train_summary']
    best_ms = result['best_params']['min_size']
    best_width = result['best_params']['width']
    if 'jump' in summary.columns:
        summary = summary[summary['jump'] == result['best_params'].get('jump', 1)]
    summary = summary[(summary['min_size'] == best_ms) & (summary['width'] == best_width)].sort_values('pen')

    n_train = len(result['train_idx'])
    se = summary['std_f1'] / np.sqrt(n_train)

    best_row_idx = summary['mean_f1'].idxmax()
    best_f1 = summary.loc[best_row_idx, 'mean_f1']
    best_se = se.loc[best_row_idx]
    threshold = best_f1 - best_se

    candidates = summary[summary['mean_f1'] >= threshold]
    chosen = candidates.loc[candidates['pen'].idxmax()]

    print(f"{cost}: argmax pen={summary.loc[best_row_idx, 'pen']:.3g} (train_f1={best_f1:.3f} +/- {best_se:.3f})")
    print(f"{cost}: 1-SE pen={chosen['pen']:.3g} (train_f1={chosen['mean_f1']:.3f}) "
          f"-- largest pen within 1 SE of the max, train-only")
    return chosen

In [ ]:
def width_sweep_test(cost, width_grid, pen, min_size, test_idx, extra_params=None):
    """Sweep width at a FIXED pen/min_size (test seeds only) -- mirrors pen_sweep_test but
    along the width axis instead, holding pen and min_size at their train-selected winning
    values. Shows how f1/precision/recall respond to width alone."""
    extra_params = extra_params or {}
    jobs = [(width, s) for width in width_grid for s in test_idx]
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(dict(cost=cost, pen=pen, min_size=min_size, width=width, **extra_params), s)
        for width, s in jobs
    )
    rows = []
    for (width, s), res in zip(jobs, scored):
        rows.append(dict(cost=cost, pen=pen, min_size=min_size, width=width, seed=int(s),
                          split='width_sweep', **res))
    sweep_raw = pd.DataFrame(rows)
    sweep_summary = (sweep_raw.groupby('width')
                      .agg(mean_precision=('precision', 'mean'), mean_recall=('recall', 'mean'),
                           mean_f1=('f1', 'mean'), mean_fp=('false_positives', 'mean'))
                      .reset_index())
    return sweep_summary, sweep_raw


def plot_width_sweep_test(cost, result, width_grid):
    """f1-vs-width (left) and the precision/recall tradeoff (right) from sweeping width on
    TEST seeds, with pen and min_size fixed at their train-selected winning values -- title
    states exactly which pen/min_size the sweep is fixed at."""
    best = result['best_params']
    extra = {}
    if 'jump' in best:
        extra['jump'] = best['jump']

    sweep_summary, sweep_raw = width_sweep_test(cost, width_grid, best['pen'], best['min_size'],
                                                 result['test_idx'], extra_params=extra)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    ax = axes[0]
    ax.plot(sweep_summary['width'], sweep_summary['mean_f1'], color='#2a78d6', marker='o', markersize=4)
    ax.axvline(best['width'], color='#e34948', linestyle='--', linewidth=1,
               label=f"winning width={best['width']:.0f}")
    ax.set_xlabel('width')
    ax.set_ylabel('mean_f1 (test)')
    ax.legend(frameon=False, fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(color='#e1e0d9', linewidth=0.6)
    ax.set_axisbelow(True)

    ax = axes[1]
    ax.plot(sweep_summary['mean_recall'], sweep_summary['mean_precision'], color='#2a78d6',
             linewidth=1.2, marker='o', markersize=4, zorder=2)
    win_row = sweep_summary[sweep_summary['width'] == best['width']]
    if not win_row.empty:
        ax.scatter(win_row['mean_recall'], win_row['mean_precision'], color='#e34948',
                   s=90, marker='*', zorder=5, label='winning width')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('recall')
    ax.set_ylabel('precision')
    ax.legend(frameon=False, fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(color='#e1e0d9', linewidth=0.6)
    ax.set_axisbelow(True)

    fig.suptitle(f"{cost}: TEST width sweep (pen={best['pen']:.3g}, min_size={best['min_size']:.0f} fixed)",
                 fontsize=10)
    plt.tight_layout()
    plt.show()

    return sweep_summary

## Section 7 -- Run the grid, one cost function per cell block

`pen`/`min_size` ranges below mirror Binseg/Pelt/BottomUp's converged ranges (that scale is a
property of the cost function, not the search algorithm). `width` ranges are untuned first passes
(see Section-0 note) -- widen `WIDTHS_<cost>` and rerun if `on_edge` fires on `width`.

- **`mahalanobis`**: for a univariate series it fits `metric = 1/variance(signal)` as a single
  global scalar (checked in `ruptures`' `CostMl` source), so the segment cost is just L2 rescaled
  by a constant -- it doesn't add anything L2 doesn't already capture. Its real purpose
  (covariance across variables) is moot with one variable. Kept in the grid anyway since a
  rescaled-L2 result is still a legitimate data point for comparison.

`cosine` has been dropped from this notebook (and Binseg/Pelt/BottomUp): for a univariate series
it structurally collapses to a sign comparison between two 1-D vectors (0 deg or 180 deg, blind to
magnitude), so it never adds information the other cost functions don't already capture.

### cost = 'ar'

NOTE: for Binseg/Pelt/BottomUp, `ar` came back with zero detections across every tested combo -- its cost targets changes in autoregressive dynamics, not mean level, and these transitions are pure mean shifts in red noise, so the same degenerate all-zero result is expected here too.

In [ ]:
PENS_ar = logrange(1, 100, 8)
MIN_SIZES_ar = MIN_SIZES
WIDTHS_ar = WIDTHS

result_ar = run_one_cost('ar', PENS_ar, MIN_SIZES_ar, WIDTHS_ar)

In [ ]:
plot_train_surface('ar', result_ar)
plot_test_pr('ar', result_ar)
plot_width_profile('ar', result_ar)
plot_parallel_coordinates('ar', result_ar)
plot_3d_scatter('ar', result_ar)

### cost = 'l1'

In [ ]:
PENS_l1 = logrange(0.01, 200, 14)
MIN_SIZES_l1 = MIN_SIZES
WIDTHS_l1 = WIDTHS

result_l1 = run_one_cost('l1', PENS_l1, MIN_SIZES_l1, WIDTHS_l1)

In [ ]:
plot_train_surface('l1', result_l1)
plot_test_pr('l1', result_l1)
plot_width_profile('l1', result_l1)
plot_min_size_profile('l1', result_l1)
plot_parallel_coordinates('l1', result_l1)
plot_3d_scatter('l1', result_l1)

### cost = 'l2'

In [ ]:
PENS_l2 = logrange(0.05, 2000, 14)
MIN_SIZES_l2 = MIN_SIZES
WIDTHS_l2 = WIDTHS

result_l2 = run_one_cost('l2', PENS_l2, MIN_SIZES_l2, WIDTHS_l2)

In [ ]:
plot_train_surface('l2', result_l2)
plot_test_pr('l2', result_l2)
plot_width_profile('l2', result_l2)
plot_min_size_profile('l2', result_l2)
plot_pen_profile('l2', result_l2)
pen_1se_selection('l2', result_l2)
plot_width_sweep_test('l2', result_l2, WIDTHS_l2)
plot_parallel_coordinates('l2', result_l2)
plot_3d_scatter('l2', result_l2)

### cost = 'linear'

NOTE: for Binseg/Pelt/BottomUp, `linear` also came back with zero detections across every combo -- its cost targets a regression relationship between a response and covariates, which doesn't exist in this univariate series.

In [ ]:
PENS_linear = logrange(1, 100, 8)
MIN_SIZES_linear = MIN_SIZES
WIDTHS_linear = WIDTHS

result_linear = run_one_cost('linear', PENS_linear, MIN_SIZES_linear, WIDTHS_linear)

In [ ]:
plot_train_surface('linear', result_linear)
plot_test_pr('linear', result_linear)
plot_width_profile('linear', result_linear)
plot_parallel_coordinates('linear', result_linear)
plot_3d_scatter('linear', result_linear)

### cost = 'normal'

In [ ]:
PENS_normal = logrange(1, 200, 12)
MIN_SIZES_normal = MIN_SIZES
WIDTHS_normal = WIDTHS

result_normal = run_one_cost('normal', PENS_normal, MIN_SIZES_normal, WIDTHS_normal)

In [ ]:
plot_train_surface('normal', result_normal)
plot_test_pr('normal', result_normal)
plot_width_profile('normal', result_normal)
plot_parallel_coordinates('normal', result_normal)
plot_3d_scatter('normal', result_normal)

### cost = 'rank'

In [ ]:
PENS_rank = logrange(1, 100, 8)
MIN_SIZES_rank = MIN_SIZES
WIDTHS_rank = WIDTHS

result_rank = run_one_cost('rank', PENS_rank, MIN_SIZES_rank, WIDTHS_rank)

In [ ]:
plot_train_surface('rank', result_rank)
plot_test_pr('rank', result_rank)
plot_width_profile('rank', result_rank)
plot_parallel_coordinates('rank', result_rank)
plot_3d_scatter('rank', result_rank)

### cost = 'rbf' (HEAVY -- reduced seed budget + coarser width grid)

HEAVY -- ~100-1000x slower per call than every other cost function, smaller seed budget (10 seeds: 7 train / 3 test), and a coarser `width` grid (4 points instead of 6) since the width axis multiplies onto an already-slow cell.

In [ ]:
PENS_rbf = logrange(1, 100, 8)
MIN_SIZES_rbf = MIN_SIZES
WIDTHS_rbf = [5, 20, 60, 150]

result_rbf = run_one_cost('rbf', PENS_rbf, MIN_SIZES_rbf, WIDTHS_rbf)

In [ ]:
plot_train_surface('rbf', result_rbf)
plot_test_pr('rbf', result_rbf)
plot_width_profile('rbf', result_rbf)
plot_parallel_coordinates('rbf', result_rbf)
plot_3d_scatter('rbf', result_rbf)

### cost = 'clinear'

In [ ]:
PENS_clinear = logrange(0.05, 200, 12)
MIN_SIZES_clinear = MIN_SIZES
WIDTHS_clinear = WIDTHS

result_clinear = run_one_cost('clinear', PENS_clinear, MIN_SIZES_clinear, WIDTHS_clinear)

In [ ]:
plot_train_surface('clinear', result_clinear)
plot_test_pr('clinear', result_clinear)
plot_width_profile('clinear', result_clinear)
plot_parallel_coordinates('clinear', result_clinear)
plot_3d_scatter('clinear', result_clinear)

### cost = 'mahalanobis'

In [ ]:
PENS_mahalanobis = logrange(1, 200, 12)
MIN_SIZES_mahalanobis = MIN_SIZES
WIDTHS_mahalanobis = WIDTHS

result_mahalanobis = run_one_cost('mahalanobis', PENS_mahalanobis, MIN_SIZES_mahalanobis, WIDTHS_mahalanobis)

In [ ]:
plot_train_surface('mahalanobis', result_mahalanobis)
plot_test_pr('mahalanobis', result_mahalanobis)
plot_width_profile('mahalanobis', result_mahalanobis)
plot_parallel_coordinates('mahalanobis', result_mahalanobis)
plot_3d_scatter('mahalanobis', result_mahalanobis)

## Section 8 -- Overview: all costs' test PR curves together

Run once every cost above has been run at least once (checks `results` has all 9 -- if you only
reran a subset, this still works off whatever's currently in `results` from earlier in the kernel
session).

In [ ]:
def plot_test_pr_grid(results, ncols=4):
    costs = list(results.keys())
    nrows = int(np.ceil(len(costs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 3.2 * nrows), squeeze=False)

    for idx, cost in enumerate(costs):
        ax = axes[idx // ncols][idx % ncols]
        res = results[cost]
        sweep = res['sweep_summary']
        row = res['conclusion_row']

        ax.plot(sweep['mean_recall'], sweep['mean_precision'], color='#2a78d6',
                 linewidth=1.5, marker='o', markersize=3, zorder=2)
        ax.scatter([row['test_recall']], [row['test_precision']], color='#e34948',
                   s=80, marker='*', zorder=5, label='selected pen')

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_title(cost, fontsize=10)
        ax.set_xlabel('recall', fontsize=7)
        ax.set_ylabel('precision', fontsize=7)
        ax.tick_params(labelsize=6)
        ax.grid(color='#e1e0d9', linewidth=0.6, zorder=0)
        ax.set_axisbelow(True)

    for j in range(len(costs), nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')

    fig.subplots_adjust(hspace=0.7, wspace=0.5, top=0.92)
    fig.suptitle("TEST set: precision/recall tradeoff (pen swept at winning min_size/width, test seeds only)", y=0.98)
    fig.savefig("window_grid_red_snr3_test_pr_grid.png", dpi=200, bbox_inches='tight')
    plt.show()


plot_test_pr_grid(results)

## Section 9 -- Save raw / summary / conclusion CSVs

In [ ]:
# raw: every scored (param, seed) row across every cost function, tagged by split --
# 'train' (full grid on the train pool, what selection used), 'test' (the single
# winning combo on the test pool), 'test_sweep' (pen swept at the winning min_size/width
# on the test pool, for the PR-curve plots above)
raw_frames = []
for cost, result in results.items():
    raw_frames.append(result['train_raw'])
    raw_frames.append(result['test_raw'])
    raw_frames.append(result['sweep_raw'])
raw_df = pd.concat(raw_frames, ignore_index=True, sort=False)
RAW_CSV = "window_grid_red_snr3_raw.csv"
raw_df.to_csv(RAW_CSV, index=False)
print(f"Saved {len(raw_df)} rows to {RAW_CSV}")

# summary: the TRAIN grid summary per cost function (what selection was based on) --
# this is what the Section 7 surface/profile/parallel-coords/scatter plots are drawn from
summary_df = pd.concat([r['train_summary'] for r in results.values()], ignore_index=True, sort=False)
SUMMARY_CSV = "window_grid_red_snr3_summary.csv"
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f"Saved {len(summary_df)} rows to {SUMMARY_CSV}")

# conclusion: one row per cost function -- train_f1 (selection score, optimistic)
# next to test_f1 (the honest, held-out score) so the gap between them is visible
conclusion_df = (pd.DataFrame([r['conclusion_row'] for r in results.values()])
                  .sort_values('test_f1', ascending=False).reset_index(drop=True))
CONCLUSION_CSV = "window_grid_red_snr3_conclusion.csv"
conclusion_df.to_csv(CONCLUSION_CSV, index=False)
print(f"Saved {len(conclusion_df)} rows to {CONCLUSION_CSV}")

In [ ]:
print("=" * 92)
print("CONCLUSION: Window on red/snr3 -- train/test split, optimized independently per cost function")
print("=" * 92)
for _, row in conclusion_df.iterrows():
    edge_note = "  <-- ON GRID EDGE: widen this cost's range and rerun" if row['on_edge'] else ""
    gap = row['train_f1'] - row['test_f1']
    print(f"{row['cost']:>12s} [{row['seed_budget']:5s}]: pen={row['best_pen']:>8.3g}  min_size={row['best_min_size']:>3.0f}  "
          f"width={row['best_width']:>4.0f}  train_f1={row['train_f1']:.3f}  test_f1={row['test_f1']:.3f}  (gap={gap:+.3f}){edge_note}")

n_edge = int(conclusion_df['on_edge'].sum())
if n_edge:
    print(f"\n{n_edge} cost function(s) did NOT converge on the train grid (optimum on the tested "
          f"boundary) -- widen those ranges and rerun before trusting their numbers.")
else:
    print(f"\nAll {len(conclusion_df)} cost functions converged (train optimum interior to their tested range).")

best_overall = conclusion_df.iloc[0]
print(f"\nBest cost function by TEST f1: '{best_overall['cost']}' "
      f"(pen={best_overall['best_pen']:.3g}, min_size={best_overall['best_min_size']:.0f}, "
      f"width={best_overall['best_width']:.0f}, test_f1={best_overall['test_f1']:.3f})")